In [14]:
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo
import json
import sys

import certifi
import pandas as pd
import requests

from azure.core.credentials import AzureNamedKeyCredential
from azure.storage.blob import BlobServiceClient
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

try:
    from IPython.display import display
except ImportError:
    display = print


# ============================================================
# CONFIGURAÇÕES
# ============================================================

OVERWRITE = True

FUSO_HORARIO = ZoneInfo(
    "America/Sao_Paulo"
)

# A data é gerada somente uma vez por execução.
DATA_EXECUCAO = datetime.now(
    FUSO_HORARIO
).strftime("%Y-%m-%d")


ARQUIVOS = [
    {
        "titulo": "PIB municipal",
        "nome": "pib_municipal.json",
        "url": (
            "https://apisidra.ibge.gov.br/"
            "values/t/5938/n6/all/"
            "v/37,516,520,528,6574/p/2021"
        ),
    },
    {
        "titulo": "Censo 2022",
        "nome": "censo_2022.json",
        "url": (
            "https://apisidra.ibge.gov.br/"
            "values/t/4714/n6/all/"
            "v/93,6318,614/p/2022"
        ),
    },
]


# ============================================================
# IDENTIFICAR O AMBIENTE
# ============================================================

def detectar_databricks() -> bool:
    try:
        dbutils.secrets
        return True

    except (
        NameError,
        AttributeError,
    ):
        return False


def localizar_raiz_projeto() -> Path:
    diretorio_atual = Path.cwd().resolve()

    for candidato in [
        diretorio_atual,
        *diretorio_atual.parents,
    ]:
        if (
            candidato
            / "src"
            / "config"
        ).is_dir():
            return candidato

    raise FileNotFoundError(
        "A pasta src/config não foi encontrada "
        "no diretório atual nem nos diretórios superiores."
    )


def exigir_valor(
    valor,
    nome: str,
) -> str:
    valor_normalizado = (
        str(valor).strip()
        if valor is not None
        else ""
    )

    if not valor_normalizado:
        raise RuntimeError(
            f"{nome} não configurado."
        )

    return valor_normalizado


IS_DATABRICKS = detectar_databricks()
BASE_DIR = localizar_raiz_projeto()

if str(BASE_DIR) not in sys.path:
    sys.path.insert(
        0,
        str(BASE_DIR),
    )


# ============================================================
# CARREGAR .ENV LOCAL
# ============================================================

if not IS_DATABRICKS:
    from dotenv import load_dotenv

    ENV_PATH = BASE_DIR / ".env"

    if not ENV_PATH.is_file():
        raise FileNotFoundError(
            f"Arquivo .env não encontrado: "
            f"{ENV_PATH}"
        )

    load_dotenv(
        dotenv_path=ENV_PATH,
        override=True,
    )


from src.config.secrets import get_secret


# ============================================================
# CONFIGURAÇÕES DO AZURE
# ============================================================

AZURE_STORAGE_ACCOUNT = exigir_valor(
    get_secret(
        secret_name="AZURE-STORAGE-ACCOUNT",
        env_name="AZURE_STORAGE_ACCOUNT",
        required=True,
    ),
    "AZURE_STORAGE_ACCOUNT",
)

AZURE_STORAGE_KEY = exigir_valor(
    get_secret(
        secret_name="AZURE-STORAGE-KEY",
        env_name="AZURE_STORAGE_KEY",
        required=True,
    ),
    "AZURE_STORAGE_KEY",
)

BRONZE_CONTAINER = exigir_valor(
    get_secret(
        secret_name="AZURE-CONTAINER-BRONZE",
        env_name="AZURE_CONTAINER_BRONZE",
        required=True,
    ),
    "AZURE_CONTAINER_BRONZE",
)


# ============================================================
# PASTA BRONZE LOCAL
# ============================================================

LOCAL_BRONZE_PATH = (
    BASE_DIR
    / "dados"
    / "bronze"
)

LOCAL_BRONZE_PATH.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# GERAR NOME DATADO
# ============================================================

def gerar_nome_datado(
    nome_original: str,
) -> str:
    prefixo_esperado = (
        f"{DATA_EXECUCAO}_"
    )

    # Evita inserir a mesma data duas vezes.
    if nome_original.startswith(
        prefixo_esperado
    ):
        return nome_original

    return (
        f"{DATA_EXECUCAO}_"
        f"{nome_original}"
    )


# ============================================================
# VALIDAR CONFIGURAÇÕES
# ============================================================

def validar_configuracao() -> None:
    if not ARQUIVOS:
        raise ValueError(
            "Nenhum arquivo foi configurado."
        )

    nomes = set()

    for indice, item in enumerate(
        ARQUIVOS,
        start=1,
    ):
        titulo = str(
            item.get("titulo", "")
        ).strip()

        nome = str(
            item.get("nome", "")
        ).strip()

        url = str(
            item.get("url", "")
        ).strip()

        if not titulo:
            raise ValueError(
                f"Entrada {indice} sem título."
            )

        if not nome:
            raise ValueError(
                f"Entrada {indice} sem nome."
            )

        if not nome.lower().endswith(
            ".json"
        ):
            raise ValueError(
                f"O arquivo deve possuir "
                f"a extensão .json: {nome}"
            )

        if not url:
            raise ValueError(
                f"Entrada {indice} sem URL."
            )

        if nome in nomes:
            raise ValueError(
                f"Nome duplicado: {nome}"
            )

        if not url.startswith(
            (
                "http://",
                "https://",
            )
        ):
            raise ValueError(
                f"URL inválida para {nome}: "
                f"{url}"
            )

        if "<a " in url or "</a>" in url:
            raise ValueError(
                f"A URL de {nome} contém HTML."
            )

        nomes.add(
            nome
        )


# ============================================================
# SESSÃO HTTP
# ============================================================

def criar_sessao_http() -> requests.Session:
    retry = Retry(
        total=5,
        connect=5,
        read=5,
        status=5,
        backoff_factor=2,
        status_forcelist=[
            429,
            500,
            502,
            503,
            504,
        ],
        allowed_methods=frozenset([
            "GET",
        ]),
        raise_on_status=False,
    )

    adapter = HTTPAdapter(
        max_retries=retry
    )

    session = requests.Session()

    session.mount(
        "https://",
        adapter,
    )

    session.mount(
        "http://",
        adapter,
    )

    session.headers.update({
        "User-Agent": (
            "Mozilla/5.0 "
            "IBGE-Azure-Pipeline/1.0"
        ),
        "Accept": "application/json,*/*",
        "Connection": "close",
    })

    return session


# ============================================================
# CLIENTE AZURE BLOB
# ============================================================

def criar_container_client():
    credential = AzureNamedKeyCredential(
        name=AZURE_STORAGE_ACCOUNT,
        key=AZURE_STORAGE_KEY,
    )

    account_url = (
        f"https://{AZURE_STORAGE_ACCOUNT}"
        ".blob.core.windows.net"
    )

    blob_service_client = BlobServiceClient(
        account_url=account_url,
        credential=credential,
    )

    blob_service_client.get_account_information()

    container_client = (
        blob_service_client
        .get_container_client(
            container=BRONZE_CONTAINER
        )
    )

    container_client.get_container_properties()

    print(
        "✅ Cliente Azure Blob autenticado."
    )

    print(
        "✅ Ambiente:",
        (
            "Databricks"
            if IS_DATABRICKS
            else "local"
        ),
    )

    print(
        f"✅ Storage Account: "
        f"{AZURE_STORAGE_ACCOUNT}"
    )

    print(
        f"✅ Contêiner Bronze: "
        f"{BRONZE_CONTAINER}"
    )

    print(
        f"✅ Pasta Bronze local: "
        f"{LOCAL_BRONZE_PATH.resolve()}"
    )

    print(
        f"✅ Data da execução: "
        f"{DATA_EXECUCAO}"
    )

    return container_client


# ============================================================
# DOWNLOAD DO ARQUIVO
# ============================================================

def baixar_arquivo(
    session: requests.Session,
    url: str,
    destino: Path,
) -> Path:
    destino = Path(
        destino
    )

    destino.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    arquivo_parcial = destino.with_suffix(
        destino.suffix + ".part"
    )

    if arquivo_parcial.exists():
        arquivo_parcial.unlink()

    try:
        with session.get(
            url=url,
            stream=True,
            verify=certifi.where(),
            timeout=(60, 3600),
            allow_redirects=True,
        ) as response:
            response.raise_for_status()

            content_type = (
                response.headers
                .get(
                    "Content-Type",
                    "",
                )
                .lower()
            )

            if "text/html" in content_type:
                raise RuntimeError(
                    "O servidor retornou HTML "
                    "em vez de JSON. "
                    f"URL final: {response.url}"
                )

            with arquivo_parcial.open(
                mode="wb"
            ) as arquivo:
                for chunk in response.iter_content(
                    chunk_size=1024 * 1024
                ):
                    if chunk:
                        arquivo.write(
                            chunk
                        )

        if not arquivo_parcial.is_file():
            raise RuntimeError(
                "Arquivo parcial não foi criado."
            )

        tamanho = (
            arquivo_parcial.stat().st_size
        )

        if tamanho <= 0:
            raise RuntimeError(
                "O download produziu "
                "um arquivo vazio."
            )

        arquivo_parcial.replace(
            destino
        )

        print(
            f"✅ Download concluído: "
            f"{destino.name}"
        )

        print(
            f"✅ Arquivo local: "
            f"{destino.resolve()}"
        )

        print(
            f"✅ Tamanho local: "
            f"{tamanho / 1024:.2f} KB"
        )

        return destino

    except Exception:
        if arquivo_parcial.exists():
            arquivo_parcial.unlink()

        raise


# ============================================================
# ENVIAR PARA O AZURE
# ============================================================

def enviar_para_bronze(
    container_client,
    arquivo_local: Path,
) -> dict:
    arquivo_local = Path(
        arquivo_local
    )

    if not arquivo_local.is_file():
        raise FileNotFoundError(
            f"Arquivo local não encontrado: "
            f"{arquivo_local}"
        )

    tamanho_local = (
        arquivo_local.stat().st_size
    )

    if tamanho_local <= 0:
        raise RuntimeError(
            f"Arquivo local vazio: "
            f"{arquivo_local}"
        )

    # O arquivo local já possui YYYY-MM-DD.
    # Portanto, o mesmo nome é utilizado no Azure.
    blob_name = arquivo_local.name

    blob_client = (
        container_client.get_blob_client(
            blob=blob_name
        )
    )

    with arquivo_local.open(
        mode="rb"
    ) as arquivo:
        blob_client.upload_blob(
            data=arquivo,
            overwrite=OVERWRITE,
        )

    propriedades = (
        blob_client.get_blob_properties()
    )

    if propriedades.size != tamanho_local:
        raise RuntimeError(
            "O tamanho enviado não corresponde "
            "ao arquivo local. "
            f"Local: {tamanho_local} bytes. "
            f"Azure: {propriedades.size} bytes."
        )

    print(
        f"✅ Upload concluído: "
        f"{blob_name}"
    )

    print(
        f"✅ Tamanho no Azure: "
        f"{propriedades.size / 1024:.2f} KB"
    )

    return {
        "blob": blob_name,
        "arquivo_local": str(
            arquivo_local.resolve()
        ),
        "size_bytes": propriedades.size,
        "etag": str(
            propriedades.etag
        ),
        "last_modified": str(
            propriedades.last_modified
        ),
    }


# ============================================================
# CARREGAR JSON DO SIDRA PARA VISUALIZAÇÃO
# ============================================================

def carregar_json_sidra(
    caminho: Path,
) -> pd.DataFrame:
    caminho = Path(
        caminho
    )

    if not caminho.is_file():
        raise FileNotFoundError(
            f"Arquivo não encontrado: "
            f"{caminho}"
        )

    with caminho.open(
        mode="r",
        encoding="utf-8-sig",
    ) as arquivo:
        dados = json.load(
            arquivo
        )

    if not isinstance(
        dados,
        list,
    ):
        raise TypeError(
            f"O arquivo {caminho.name} "
            "não contém uma lista JSON."
        )

    if not dados:
        raise ValueError(
            f"O arquivo {caminho.name} "
            "está vazio."
        )

    if not all(
        isinstance(registro, dict)
        for registro in dados
    ):
        raise TypeError(
            f"O arquivo {caminho.name} contém "
            "registros em formato inesperado."
        )

    primeiro_registro = dados[0]

    termos_cabecalho = [
        "município",
        "municipio",
        "variável",
        "variavel",
        "valor",
        "unidade",
        "ano",
        "nível territorial",
        "nivel territorial",
    ]

    parece_cabecalho_sidra = (
        primeiro_registro
        and all(
            isinstance(valor, str)
            for valor in primeiro_registro.values()
        )
        and any(
            termo in str(valor).lower()
            for valor in primeiro_registro.values()
            for termo in termos_cabecalho
        )
    )

    if parece_cabecalho_sidra:
        dataframe = pd.DataFrame(
            dados[1:]
        ).rename(
            columns=primeiro_registro
        )

    else:
        dataframe = pd.json_normalize(
            dados,
            sep="_",
        )

    dataframe.columns = [
        str(coluna).strip()
        for coluna in dataframe.columns
    ]

    return dataframe


# ============================================================
# EXECUTAR PIPELINE
# ============================================================

def executar_pipeline():
    validar_configuracao()

    container_client = (
        criar_container_client()
    )

    session = criar_sessao_http()

    uploads_concluidos = []
    falhas = []

    try:
        total = len(
            ARQUIVOS
        )

        for indice, item in enumerate(
            ARQUIVOS,
            start=1,
        ):
            titulo = item["titulo"]
            nome_original = item["nome"]
            url = item["url"]

            # O nome é datado uma única vez aqui.
            nome_datado = gerar_nome_datado(
                nome_original
            )

            destino_local = (
                LOCAL_BRONZE_PATH
                / nome_datado
            )

            print()
            print(
                f"[{indice}/{total}] "
                f"Processando: {nome_datado}"
            )

            try:
                arquivo_baixado = baixar_arquivo(
                    session=session,
                    url=url,
                    destino=destino_local,
                )

                resultado = enviar_para_bronze(
                    container_client=container_client,
                    arquivo_local=arquivo_baixado,
                )

                uploads_concluidos.append({
                    "titulo": titulo,
                    "arquivo_original": nome_original,
                    "arquivo": nome_datado,
                    "origem": url,
                    "container": BRONZE_CONTAINER,
                    **resultado,
                })

            except Exception as exc:
                erro = (
                    f"{type(exc).__name__}: "
                    f"{exc}"
                )

                falhas.append({
                    "titulo": titulo,
                    "arquivo": nome_datado,
                    "origem": url,
                    "blob": nome_datado,
                    "erro": erro,
                })

                print(
                    f"❌ Falha em "
                    f"{nome_datado}: "
                    f"{erro}"
                )

    finally:
        session.close()

    print()
    print(
        "========== RESUMO =========="
    )

    print(
        f"✅ Arquivos enviados: "
        f"{len(uploads_concluidos)}"
    )

    print(
        f"❌ Falhas: "
        f"{len(falhas)}"
    )

    print(
        f"✅ Contêiner: "
        f"{BRONZE_CONTAINER}"
    )

    print(
        f"✅ Pasta local: "
        f"{LOCAL_BRONZE_PATH.resolve()}"
    )

    if falhas:
        detalhes = "\n".join(
            (
                f"- Arquivo: "
                f"{falha['arquivo']}\n"
                f"  Origem: "
                f"{falha['origem']}\n"
                f"  Blob: "
                f"{falha['blob']}\n"
                f"  Erro: "
                f"{falha['erro']}"
            )
            for falha in falhas
        )

        raise RuntimeError(
            f"{len(falhas)} arquivo(s) "
            "não foram enviados:\n"
            f"{detalhes}"
        )

    return (
        container_client,
        uploads_concluidos,
    )


# ============================================================
# VALIDAR UPLOADS
# ============================================================

def validar_bronze(
    container_client,
    uploads_concluidos,
) -> None:
    print()
    print(
        "========== VALIDAÇÃO BRONZE =========="
    )

    for upload in uploads_concluidos:
        arquivo_local = Path(
            upload["arquivo_local"]
        )

        blob_name = upload["blob"]

        tamanho_local = (
            arquivo_local.stat().st_size
        )

        propriedades = (
            container_client
            .get_blob_client(
                blob=blob_name
            )
            .get_blob_properties()
        )

        if propriedades.size != tamanho_local:
            raise RuntimeError(
                f"Tamanho divergente em "
                f"{blob_name}. "
                f"Local: {tamanho_local}. "
                f"Azure: {propriedades.size}."
            )

        print(
            f"✅ Validado: "
            f"{blob_name} "
            f"({propriedades.size / 1024:.2f} KB)"
        )

    print(
        "✅ Arquivos locais e blobs "
        "validados com sucesso."
    )


# ============================================================
# EXIBIR 2 REGISTROS DE CADA ARQUIVO DA EXECUÇÃO
# ============================================================

def exibir_amostras(
    uploads_concluidos,
) -> dict:
    print()
    print(
        "========== AMOSTRAS DOS ARQUIVOS =========="
    )

    dataframes = {}
    falhas_visualizacao = []

    for indice, upload in enumerate(
        uploads_concluidos,
        start=1,
    ):
        titulo = upload["titulo"]

        caminho = Path(
            upload["arquivo_local"]
        )

        try:
            dataframe = carregar_json_sidra(
                caminho
            )

            dataframes[
                titulo
            ] = dataframe

            print()
            print(
                "=" * 100
            )

            print(
                f"[{indice}/{len(uploads_concluidos)}] "
                f"{titulo}"
            )

            print(
                "=" * 100
            )

            print(
                f"✅ Arquivo local: "
                f"{caminho.name}"
            )

            print(
                f"✅ Blob no Azure: "
                f"{upload['blob']}"
            )

            print(
                f"✅ Registros totais: "
                f"{len(dataframe):,}"
            )

            print(
                f"✅ Colunas: "
                f"{len(dataframe.columns)}"
            )

            print(
                "✅ Exibindo 2 registros:"
            )

            display(
                dataframe.head(2)
            )

        except Exception as erro:
            mensagem = (
                f"{type(erro).__name__}: "
                f"{erro}"
            )

            falhas_visualizacao.append({
                "titulo": titulo,
                "arquivo": caminho.name,
                "erro": mensagem,
            })

            print(
                f"❌ Erro ao visualizar "
                f"{caminho.name}: "
                f"{mensagem}"
            )

    print()
    print(
        f"✅ Tabelas exibidas: "
        f"{len(dataframes)}"
    )

    print(
        f"❌ Falhas de visualização: "
        f"{len(falhas_visualizacao)}"
    )

    if falhas_visualizacao:
        for falha in falhas_visualizacao:
            print(
                f"   • {falha['titulo']} "
                f"({falha['arquivo']}): "
                f"{falha['erro']}"
            )

    return dataframes


# ============================================================
# EXECUÇÃO
# ============================================================

(
    container_client,
    uploads_concluidos,
) = executar_pipeline()

validar_bronze(
    container_client=container_client,
    uploads_concluidos=uploads_concluidos,
)

dataframes = exibir_amostras(
    uploads_concluidos=uploads_concluidos,
)

df_pib = dataframes.get(
    "PIB municipal"
)

df_censo = dataframes.get(
    "Censo 2022"
)

✅ Cliente Azure Blob autenticado.
✅ Ambiente: local
✅ Storage Account: stfiaptechchallenge2iast
✅ Contêiner Bronze: bronze
✅ Pasta Bronze local: /home/linux/projetos/postech-2iast-fase3-modelo-preditivo-g28/dados/bronze
✅ Data da execução: 2026-10-19

[1/2] Processando: 2026-10-19_pib_municipal.json
✅ Download concluído: 2026-10-19_pib_municipal.json
✅ Arquivo local: /home/linux/projetos/postech-2iast-fase3-modelo-preditivo-g28/dados/bronze/2026-10-19_pib_municipal.json
✅ Tamanho local: 10191.82 KB
✅ Upload concluído: 2026-10-19_pib_municipal.json
✅ Tamanho no Azure: 10191.82 KB

[2/2] Processando: 2026-10-19_censo_2022.json
✅ Download concluído: 2026-10-19_censo_2022.json
✅ Arquivo local: /home/linux/projetos/postech-2iast-fase3-modelo-preditivo-g28/dados/bronze/2026-10-19_censo_2022.json
✅ Tamanho local: 4542.91 KB
✅ Upload concluído: 2026-10-19_censo_2022.json
✅ Tamanho no Azure: 4542.91 KB

========== RESUMO ==========
✅ Arquivos enviados: 2
❌ Falhas: 0
✅ Contêiner: bronze
✅ Pasta 

,Nível Territorial (Código),Nível Territorial,Unidade de Medida (Código),Unidade de Medida,Valor,Município (Código),Município,Variável (Código),Variável,Ano (Código),Ano
0,6,Município,40,Mil Reais,734467,1100015,Alta Floresta D'Oeste - RO,37,Produto Interno Bruto a preços correntes,2021,2021
1,6,Município,2,%,45.48,1100015,Alta Floresta D'Oeste - RO,516,Participação do valor adicionado bruto a preço...,2021,2021



[2/2] Censo 2022
✅ Arquivo local: 2026-10-19_censo_2022.json
✅ Blob no Azure: 2026-10-19_censo_2022.json
✅ Registros totais: 16,710
✅ Colunas: 11
✅ Exibindo 2 registros:


,Nível Territorial (Código),Nível Territorial,Unidade de Medida (Código),Unidade de Medida,Valor,Município (Código),Município,Variável (Código),Variável,Ano (Código),Ano
0,6,Município,45,Pessoas,21494,1100015,Alta Floresta D'Oeste - RO,93,População residente,2022,2022
1,6,Município,26,Quilômetros quadrados,7067.127,1100015,Alta Floresta D'Oeste - RO,6318,Área da unidade territorial,2022,2022



✅ Tabelas exibidas: 2
❌ Falhas de visualização: 0
